# Logicore DB Inspector

Run queries directly against the session persistence database.
Connection stays alive across cells — no reconnect overhead.

In [ ]:
import os, sys, json
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, Markdown

sys.path.insert(0, str(Path.cwd().parent))

# ── Auto-detect DB from env / settings ───────────────────────────
from dotenv import load_dotenv
load_dotenv()

DB_URL = os.getenv("STORAGE_DB_URL", "")
if not DB_URL:
    try:
        from logicore.config import settings
        DB_URL = settings.STORAGE_DB_URL
    except Exception:
        pass

if not DB_URL:
    base = Path.home() / ".logicore"
    DB_URL = f"sqlite:///{base / 'database' / 'logicore.db'}"

IS_PG = DB_URL.startswith(("postgresql://", "postgres://"))
print(f"DB: {DB_URL[:50]}{'...' if len(DB_URL) > 50 else ''}")
print(f"Type: {'PostgreSQL' if IS_PG else 'SQLite'}")

In [ ]:
# ── Persistent connection (stays alive across cells) ─────────────
if IS_PG:
    import psycopg2, psycopg2.extras
    CONN = psycopg2.connect(DB_URL)
    CONN.autocommit = True
    def query(sql, params=None):
        with CONN.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql, params)
            if cur.description:
                return [dict(r) for r in cur.fetchall()]
            return []
    def execute(sql, params=None):
        with CONN.cursor() as cur:
            cur.execute(sql, params)
            CONN.commit()
            return cur.rowcount
else:
    import sqlite3
    db_path = DB_URL.replace("sqlite:///", "")
    CONN = sqlite3.connect(db_path)
    CONN.row_factory = sqlite3.Row
    CONN.execute("PRAGMA journal_mode=WAL")
    def query(sql, params=None):
        cur = CONN.execute(sql, params or ())
        if cur.description:
            return [dict(r) for r in cur.fetchall()]
        return []
    def execute(sql, params=None):
        cur = CONN.execute(sql, params or ())
        CONN.commit()
        return cur.rowcount

print("Connection open.")
print("Use:  query('SELECT ...')  — returns list of dicts")
print("      execute('INSERT ...') — returns rowcount")

---
## Schema & Table List

In [ ]:
if IS_PG:
    tables = query("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name
    """)
else:
    tables = query("SELECT name AS table_name FROM sqlite_master WHERE type='table' ORDER BY name")

display(Markdown("### Tables"))
for t in tables:
    count = query(f"SELECT COUNT(*) AS c FROM {t['table_name']}")[0]['c']
    print(f"  {t['table_name']:30s}  {count} rows")

---
## Sessions

In [ ]:
rows = query("""
    SELECT session_id, provider, model, revision, has_attachments,
           description, created_at, updated_at,
           length(session_messages) AS msg_bytes
    FROM sessions
    ORDER BY updated_at DESC
    LIMIT 20
""")
display(Markdown(f"### Sessions ({len(rows)} shown)"))
for r in rows:
    print(json.dumps(r, indent=2, default=str))
    print()

In [ ]:
# ── Peek at messages for a specific session ──────────────────────
SESSION_ID = rows[0]['session_id'] if rows else ''
print(f"Session: {SESSION_ID}\n")

if SESSION_ID:
    if IS_PG:
        r = query("SELECT session_messages FROM sessions WHERE session_id = %s", (SESSION_ID,))
        msgs = r[0]['session_messages'] if r else []
    else:
        r = query("SELECT session_messages FROM sessions WHERE session_id = ?", (SESSION_ID,))
        msgs = json.loads(r[0]['session_messages']) if r else []

    for m in msgs:
        role = m.get('role', '?')
        content = str(m.get('content', ''))[:120]
        tc = m.get('tool_calls')
        tool_info = f"  [tool_calls: {len(tc)}]" if tc else ""
        print(f"  {role:12s} {content}{tool_info}")

In [ ]:
# ── Peek at metadata (context column) ────────────────────────────
if SESSION_ID:
    r = query("SELECT context FROM sessions WHERE session_id = %s" if IS_PG
              else "SELECT context FROM sessions WHERE session_id = ?", (SESSION_ID,))
    ctx = r[0]['context'] if r else ''
    if ctx:
        print(json.dumps(json.loads(ctx), indent=2, default=str))
    else:
        print("(empty)")

---
## Telemetry

In [ ]:
rows = query("""
    SELECT t.session_id,
           t.input_tokens, t.output_tokens,
           t.cache_read_tokens, t.cache_write_tokens,
           t.reasoning_tokens, t.tool_calls, t.api_calls,
           s.provider, s.model
    FROM session_telemetry t
    JOIN sessions s ON s.session_id = t.session_id
    ORDER BY s.updated_at DESC
    LIMIT 20
""")
display(Markdown(f"### Telemetry ({len(rows)} shown)"))
for r in rows:
    print(json.dumps(r, indent=2, default=str))
    print()

---
## Pending Snapshot Syncs

In [ ]:
rows = query("SELECT * FROM pending_syncs ORDER BY enqueued_at DESC")
display(Markdown(f"### Pending Syncs ({len(rows)})"))
for r in rows:
    print(json.dumps(r, indent=2, default=str))

---
## Schema Version

In [ ]:
rows = query("SELECT * FROM schema_version ORDER BY version DESC")
for r in rows:
    print(r)

---
## Raw SQL Runner

Edit the query below and run the cell.

In [ ]:
rows = query("SELECT * FROM sessions LIMIT 5")
for r in rows:
    print(json.dumps(r, indent=2, default=str))
    print()

---
## Cleanup

In [ ]:
CONN.close()
print("Connection closed.")